In [3]:
import re
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
from matplotlib.backends.backend_pdf import PdfPages

# ==============================================================
# 1. Lecture du fichier et extraction des HOMO, LUMO, GAP
# ==============================================================

def parse_marche(filename):
    with open(filename, 'r', encoding='utf-8') as f:
        content = f.read()

    # Recherche des énergies
    pattern = r"HOMO:\s*([-+]?\d*\.\d+|\d+)\s*Ha\s*LUMO:\s*([-+]?\d*\.\d+|\d+)\s*Ha\s*GAP:\s*([-+]?\d*\.\d+|\d+)\s*eV"
    data = re.findall(pattern, content)

    homo = np.array([float(x[0]) * 27.2114 for x in data])
    lumo = np.array([float(x[1]) * 27.2114 for x in data])
    gap  = np.array([float(x[2]) for x in data])
    return homo, lumo, gap


# ==============================================================
# 2. Outils mathématiques
# ==============================================================

def fitness_auto_correlation(fitnesses: list[float], max_lag: int) -> list[float]:
    """
    Calcule les coefficients d'auto-corrélation d'une liste de valeurs fitness.
    """
    auto_correlations: list[float] = []
    for l in range(1, max_lag):
        auto_correlations.append(float(np.corrcoef(fitnesses[:-l], fitnesses[l:])[0, 1]))
    return auto_correlations


def correlation_length(C, threshold= 0.15):
    """
    Estimation robuste de Lc : premier lag où C(k) tombe en dessous du seuil.
    """
    C = np.array(C)
    for i, val in enumerate(C, start=1):
        if val < threshold:
            return i
    return len(C)


def delta_fitness(f):
    return np.diff(f)


def fitness_stats(dF, threshold=1e-3):
    inc = np.sum(dF > threshold)
    dec = np.sum(dF < -threshold)
    neutre = np.sum(np.abs(dF) <= threshold)
    total = len(dF)
    return inc/total, neutre/total, dec/total


# ============================
# 3. Analyse et sauvegarde txt + PDF
# ============================
def analyze_fitness(name, f, txt_file, pdf):
    C = fitness_auto_correlation(f , max_lag=50)
    Lcorr = correlation_length(C)
    dF = delta_fitness(f)
    inc, neutre, dec = fitness_stats(dF)

    txt_file.write(f"\n=== Analyse for {name} ===\n")
    txt_file.write(f"Correlation length : {Lcorr:.2f}\n")
    txt_file.write(f"Increasing step : {inc*100:.1f}%, Neutral : {neutre*100:.1f}%, Decreasing : {dec*100:.1f}%\n")
    txt_file.write(f"Mean Δfitness: {np.mean(np.abs(dF)):.4f}\n")

    # Graphiques
    fig, axs = plt.subplots(1, 3, figsize=(14, 4))
    axs[0].plot(f)
    axs[0].set_title(f"{name} - evolution")
    axs[0].set_xlabel("step")
    axs[0].set_ylabel(name)

    axs[1].plot(np.arange(1, len(C)+1), C)
    axs[1].set_title(f"{name} Autocorrelation")
    axs[1].set_xlabel("Lag k")
    axs[1].set_ylabel("C(k)")

    axs[2].hist(dF, bins=15, alpha=0.7)
    axs[2].set_title(f"Distribution of Δ{name}")
    axs[2].set_xlabel("Δfitness")
    axs[2].set_ylabel("Frequency")

    plt.tight_layout()
    pdf.savefig(fig)
    plt.close(fig)

    return {
        "name": name,
        "Lcorr": Lcorr,
        "inc": inc,
        "neutre": neutre,
        "dec": dec,
        "mean_dF": np.mean(np.abs(dF))
    }


# ============================
# 4. Exécution
# ============================
if __name__ == "__main__":
    homo, lumo, gap = parse_marche("marche_aleatoire.txt")

    results = []
    with open("Data_Landscape.txt", "w", encoding="utf-8") as txt_file, \
         PdfPages("Data_Landscape.pdf") as pdf:
        results.append(analyze_fitness("HOMO", homo, txt_file, pdf))
        results.append(analyze_fitness("LUMO", lumo, txt_file, pdf))
        results.append(analyze_fitness("GAP", gap, txt_file, pdf))

        # Résumé global
        txt_file.write("\n=== RÉSUMÉ GLOBAL ===\n")
        for r in results:
            txt_file.write(f"{r['name']}: Lc={r['Lcorr']:.2f}, Δmoy={r['mean_dF']:.4f}\n")

    print("Analyse complète enregistrée dans 'Data_Landscape.txt' et 'Data_Landscape.pdf'.")



Analyse complète enregistrée dans 'Data_Landscape.txt' et 'Data_Landscape.pdf'.
